# Dense Code Retrieval: tree-sitter AST chunking (cAST) + CodeRankEmbed, evaluated on CoIR **AppsRetrieval** with MTEB

**Pipeline**

1. **Pre-processing (documents):** every Python solution is parsed with **tree-sitter** and split with the cAST *split-then-merge* algorithm. Chunks from inside a big function/class get the **ancestor signature** added on top as a comment.
2. **Pre-processing (queries):** the problem statement gets CodeRankEmbed's required query prefix.
3. **Embedding:** `nomic-ai/CodeRankEmbed` (137M params), fp16 on GPU.
4. **Post-processing:** MTEB needs **one vector per document**, so the chunk vectors of each document are combined with a **length-weighted mean** and re-normalized.
5. **Evaluation:** MTEB `AppsRetrieval` (test split), main metric nDCG@10. Results saved to `appsretrieval_results.json`.

Run the cells top to bottom. Use a **GPU runtime** (Runtime → Change runtime type → T4 GPU).

## Step 1: Install packages

In [ ]:
!pip install -q "mteb>=2.2" "sentence-transformers>=3.0" tree-sitter tree-sitter-python einops

## Step 2: Imports and GPU check

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # less memory fragmentation; set before torch uses the GPU

import json
import random
import time
from collections import Counter
from dataclasses import dataclass, replace
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import mteb
from sentence_transformers import SentenceTransformer
from mteb.models.abs_encoder import AbsEncoder
from mteb.models.model_meta import ModelMeta, ScoringFunction
from mteb.models.cache_wrappers import CachedEmbeddingWrapper
from mteb.types import PromptType

import tree_sitter_python as tspy
from tree_sitter import Language, Parser

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("mteb version:", mteb.__version__)
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "no GPU (this will be slow)")

random.seed(0)
np.random.seed(0)
PY_LANGUAGE = Language(tspy.language())

## Step 2b: Save everything to Google Drive
Colab deletes local files when the runtime disconnects. Everything this notebook produces (embedding cache, rankings, result JSONs) goes into one Drive folder, so a crash or disconnect doesn't lose finished work.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    RUN_DIR = Path("/content/drive/MyDrive/code_retrieval_apps")
except ImportError:                                  # not in Colab: save next to the notebook
    RUN_DIR = Path("code_retrieval_apps")
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Saving outputs to:", RUN_DIR.resolve())

## Step 3: Config
All settings live here. Change a value and re-run from this cell.

- `max_chunk_chars` is the chunk budget, measured in **non-whitespace characters** (the cAST paper's measure). Look at the corpus statistics in Step 7 before tuning it.
- `add_ancestor_header`: add `# def solve(...)` / `# class X:` lines on top of chunks cut out of a big function/class.
- `add_imports_header`: repeat the file's import lines on every chunk (off by default).

In [ ]:
@dataclass(frozen=True)
class Config:
    # model
    model_name: str = "nomic-ai/CodeRankEmbed"
    query_prefix: str = "Represent this query for searching relevant code: "  # required by CodeRankEmbed
    query_max_seq_length: int = 2048  # tokens; cuts only 4 of 3,765 queries (Step 7b)
    doc_max_seq_length: int = 1024    # tokens; cuts only 2 of 12,614 chunks (Step 7b). Keeps document batches small in memory
    fp16: bool = True                 # half precision on GPU (ignored on CPU)
    batch_size: int = 64              # documents / chunks
    query_batch_size: int = 16        # queries are long: smaller batches avoid running out of GPU memory

    # chunking
    max_chunk_chars: int = 512        # budget in NON-whitespace characters
    add_ancestor_header: bool = True
    add_imports_header: bool = False
    header_node_types: tuple = ("function_definition", "class_definition")
    import_node_types: tuple = ("import_statement", "import_from_statement", "future_import_statement")

    # evaluation
    task_name: str = "AppsRetrieval"
    results_file: str = str(RUN_DIR / "appsretrieval_results.json")
    prediction_folder: str = str(RUN_DIR / "predictions" / "main")

    # checkpointing
    emb_cache_dir: str = str(RUN_DIR / "embedding_cache")
    cache_version: str = "v2"         # bump this (e.g. "v2") if you change the chunker/encoder CODE, so old vectors aren't reused

CFG = Config()
CFG

## Step 4: See how tree-sitter builds the tree
This parses a small example and prints every **named** node with its line range. Tokens like `def`, `(`, `:` are unnamed nodes and are hidden to keep the output readable.

In [ ]:
parser = Parser(PY_LANGUAGE)

demo_src = '''import sys

def solve(nums):
    total = 0
    for x in nums:
        if x % 2 == 0:
            total += x
    return total

n = int(input())
arr = list(map(int, input().split()))
print(solve(arr))
'''

def show_tree(node, src: bytes, depth: int = 0, max_depth: int = 6) -> None:
    # Print named nodes as an indented tree: type, line range, first line of its text.
    if depth > max_depth:
        return
    first_line = src[node.start_byte:node.end_byte].decode("utf-8", "replace").split("\n")[0]
    if len(first_line) > 40:
        first_line = first_line[:37] + "..."
    s, e = node.start_point[0] + 1, node.end_point[0] + 1
    lines = f"L{s}" if s == e else f"L{s}-{e}"
    print(f"{'  ' * depth}{node.type:<22} {lines:<7} {first_line}")
    for child in node.named_children:
        show_tree(child, src, depth + 1, max_depth)

demo_bytes = demo_src.encode("utf-8")
demo_tree = parser.parse(demo_bytes)
show_tree(demo_tree.root_node, demo_bytes, max_depth=3)
print("\nparse errors:", demo_tree.root_node.has_error)

# Old Python 2 code parses too:
py2 = b"n = int(raw_input())\nprint n * 2\n"
py2_tree = parser.parse(py2)
print("\nPython 2 example -> top-level nodes:", [c.type for c in py2_tree.root_node.children],
      "| parse errors:", py2_tree.root_node.has_error)

## Step 5: The AST chunker (cAST split-then-merge)

For one document:
1. If the whole file fits the budget → **one chunk** (the whole tree).
2. Otherwise walk the top-level nodes in order and **greedily merge** neighbours into a chunk until the next node would go over the budget.
3. A node that is **too big on its own** is **split**: we go into its body and repeat.
   - The node's own header line (`for _ in range(t):`, `def solve(a):`, `try:` …) stays **in the code of the first chunk** made from its body.
   - Parts that come after the body (`else:`, `except:`, `finally:`) are chunked too.
   - If the node is a function/class, its signature is also remembered as an *ancestor* and added as a comment on top of the **later** chunks from its body.

Chunk text is cut from the original file by byte position. Step 7 checks over the whole corpus that the chunks together contain every character of the original code.

In [ ]:
def nonws_len(text: str) -> int:
    # Chunk size = number of non-whitespace characters (cAST paper's measure).
    return sum(1 for ch in text if not ch.isspace())


class ASTChunker:
    # cAST-style split-then-merge chunker built on tree-sitter. Language-agnostic:
    # to support another language, pass its tree-sitter Language and its node type names.

    def __init__(self, language, max_chunk_chars=512, add_ancestor_header=True,
                 add_imports_header=False,
                 header_node_types=("function_definition", "class_definition"),
                 import_node_types=("import_statement", "import_from_statement")):
        self.parser = Parser(language)
        self.max_chars = max_chunk_chars
        self.add_ancestor_header = add_ancestor_header
        self.add_imports_header = add_imports_header
        self.header_types = set(header_node_types)
        self.import_types = set(import_node_types)
        self.reset_stats()

    def reset_stats(self):
        self.stats = {"docs": 0, "chunks": 0, "docs_with_parse_errors": 0}

    # ---------------- public ----------------
    def chunk(self, code: str) -> list[dict]:
        self.stats["docs"] += 1
        src = code.encode("utf-8")
        root = self.parser.parse(src).root_node
        if root.has_error:
            self.stats["docs_with_parse_errors"] += 1

        if nonws_len(code) <= self.max_chars or not root.children:
            groups = [([root], [], None)]                         # whole file fits: one chunk
        else:
            groups = self._chunk_nodes(root.children, src, ancestors=[])

        chunks = [self._make_chunk(nodes, anc, src, start) for nodes, anc, start in groups]
        chunks = [c for c in chunks if c["code"].strip()] or [self._whole(code)]

        if self.add_imports_header and len(chunks) > 1:
            imports = [self._text(n, src) for n in root.children if n.type in self.import_types]
            if imports:
                block = "\n".join(imports) + "\n"
                for c in chunks:
                    if not any(t in self.import_types for t in c["node_types"]):
                        c["text"] = block + c["text"]

        for i, c in enumerate(chunks):
            c["chunk_id"], c["n_chunks"], c["has_parse_error"] = i, len(chunks), root.has_error
        self.stats["chunks"] += len(chunks)
        return chunks

    # ---------------- split-then-merge ----------------
    def _chunk_nodes(self, nodes, src, ancestors):
        groups, cur, cur_size = [], [], 0
        for node in nodes:
            s = nonws_len(self._text(node, src))
            if (not cur and s > self.max_chars) or (cur_size + s > self.max_chars):
                if cur:                                           # close the current chunk
                    groups.append((cur, ancestors, None))
                    cur, cur_size = [], 0
                if s > self.max_chars:                            # node too big alone: split it
                    groups.extend(self._split_big_node(node, src, ancestors))
                    continue
            cur.append(node)                                      # merge into the current chunk
            cur_size += s
        if cur:
            groups.append((cur, ancestors, None))
        return groups

    @staticmethod
    def _find_body(node):
        # The indented block of a compound statement (def, class, for, while, if, with, try, else, except ...).
        for field in ("body", "consequence"):
            body = node.child_by_field_name(field)
            if body is not None:
                return body
        return next((c for c in node.children if c.type == "block"), None)

    def _children_cover(self, node, src) -> bool:
        # Some nodes (e.g. a long string) have text that is NOT inside any child node.
        # Splitting those into children would drop that text, so we only split when the children cover it all.
        return nonws_len(self._text(node, src)) == sum(nonws_len(self._text(c, src)) for c in node.children)

    def _split_big_node(self, node, src, ancestors):
        body = self._find_body(node)
        if body is not None and body.children and self._children_cover(node, src) and self._children_cover(body, src):
            new_ancestors = ancestors
            if node.type in self.header_types:                    # remember "def f(...):" / "class X:"
                header = src[node.start_byte:body.start_byte].decode("utf-8", "replace")
                new_ancestors = ancestors + [" ".join(header.split())]
            groups = self._chunk_nodes(body.children, src, new_ancestors)
            if groups:
                # The first chunk of the body starts at the node itself, so the header line
                # ("for _ in range(t):", "def f(a):", "try:") is kept as real code, not lost.
                # That header is already in its code, so it only gets the OUTER ancestors as comments.
                groups[0] = (groups[0][0], ancestors, node.start_byte)
            after = [c for c in node.children if c.start_byte >= body.end_byte]   # else / except / finally
            if after:
                groups += self._chunk_nodes(after, src, ancestors)
            return groups
        if node.children and self._children_cover(node, src):
            return self._chunk_nodes(node.children, src, ancestors)
        return [([node], ancestors, None)]                        # can't split safely: keep it whole

    # ---------------- helpers ----------------
    @staticmethod
    def _text(node, src):
        return src[node.start_byte:node.end_byte].decode("utf-8", "replace")

    def _make_chunk(self, nodes, ancestors, src, start_byte=None):
        first, last = nodes[0], nodes[-1]
        start_byte = first.start_byte if start_byte is None else start_byte
        raw = src[start_byte:last.end_byte].decode("utf-8", "replace")
        line_start = src.rfind(b"\n", 0, start_byte) + 1          # keep the first line's indentation
        indent = src[line_start:start_byte].decode("utf-8", "replace")
        code = indent + raw if indent.strip() == "" else raw
        header = "".join(f"# {a}\n" for a in ancestors) if self.add_ancestor_header else ""
        return {
            "text": header + code,          # what gets embedded
            "code": code,                   # raw code only
            "ancestors": list(ancestors),   # metadata (not embedded unless header is on)
            "start_line": src.count(b"\n", 0, start_byte) + 1,
            "end_line": max(src.count(b"\n", 0, start_byte) + 1,
                            last.end_point[0] + (1 if last.end_point[1] > 0 else 0)),
            "node_types": [n.type for n in nodes],
            "size": nonws_len(code),
        }

    @staticmethod
    def _whole(code):
        return {"text": code, "code": code, "ancestors": [], "start_line": 1,
                "end_line": max(code.count("\n"), 1), "node_types": ["module"],
                "size": nonws_len(code)}


def make_chunker(cfg: Config) -> ASTChunker:
    return ASTChunker(PY_LANGUAGE, cfg.max_chunk_chars, cfg.add_ancestor_header,
                      cfg.add_imports_header, cfg.header_node_types, cfg.import_node_types)


def code_is_complete(code: str, chunks: list[dict]) -> bool:
    # True if the chunks, in order, contain exactly the original code (ignoring whitespace).
    return "".join(code.split()) == "".join("".join(c["code"] for c in chunks).split())

## Step 6: Test the chunker
A small budget (120) forces splitting, so you can see the merge step, the split step and the ancestor headers.

In [ ]:
test_src = '''import sys
from heapq import heappush, heappop

class Solver:
    def __init__(self, n):
        self.n = n
        self.adj = [[] for _ in range(n)]

    def dijkstra(self, s):
        dist = [float("inf")] * self.n
        dist[s] = 0
        pq = [(0, s)]
        while pq:
            d, u = heappop(pq)
            if d > dist[u]:
                continue
            for v, w in self.adj[u]:
                if d + w < dist[v]:
                    dist[v] = d + w
                    heappush(pq, (dist[v], v))
        return dist

n, m = map(int, input().split())
g = Solver(n)
for _ in range(m):
    a, b, c = map(int, input().split())
    g.adj[a - 1].append((b - 1, c))
print(*g.dijkstra(0))
'''

def print_chunks(chunks, max_lines=12):
    for c in chunks:
        print(f"--- chunk {c['chunk_id'] + 1}/{c['n_chunks']} | lines {c['start_line']}-{c['end_line']} "
              f"| size {c['size']} | nodes {c['node_types']}")
        lines = c["text"].rstrip("\n").split("\n")
        print("\n".join(lines[:max_lines]) + ("\n    ..." if len(lines) > max_lines else ""))
    print()

print("===== budget = 120 (forced splitting) =====")
print_chunks(make_chunker(replace(CFG, max_chunk_chars=120)).chunk(test_src))

print(f"===== budget = {CFG.max_chunk_chars} (your config) =====")
print_chunks(make_chunker(CFG).chunk(test_src))

# A typical competitive-programming script: a big per-test-case loop with try/except and for-else.
script_src = '''import random
t=int(input())
for _ in range(t):
    n=int(input())
    arr=list(map(int,input().split()))
    total=0
    for x in arr:
        if x%2==0:
            total+=x
        else:
            total-=x
    try:
        ans = total // n
    except ZeroDivisionError:
        ans = -1
    print(ans)
else:
    print("done")
'''
print("===== script, budget = 60 =====")
script_chunks = make_chunker(replace(CFG, max_chunk_chars=60)).chunk(script_src)
print_chunks(script_chunks, max_lines=20)
print("no code lost:", code_is_complete(test_src, make_chunker(replace(CFG, max_chunk_chars=120)).chunk(test_src))
      and code_is_complete(script_src, script_chunks))

## Step 7: Load AppsRetrieval and look at the corpus
Queries are problem statements (English), documents are Python solutions. This cell also runs the chunker over the whole corpus (CPU only, fast) so you can see how many chunks a document gets with your budget.

In [ ]:
apps_task = mteb.get_task(CFG.task_name)
apps_task.load_data()
subset = next(iter(apps_task.dataset))            # "default" for this task
split = apps_task.dataset[subset]["test"]

corpus, queries, qrels = split["corpus"], split["queries"], split["relevant_docs"]
doc_text = dict(zip(corpus["id"], corpus["text"]))
query_text = dict(zip(queries["id"], queries["text"]))
print(f"subset={subset!r} | corpus: {len(doc_text)} docs | queries: {len(query_text)} | qrels: {len(qrels)} queries")

sizes = np.array([nonws_len(t) for t in doc_text.values()])
print("\nDocument size (non-whitespace chars) percentiles:")
for p in (10, 25, 50, 75, 90, 99):
    print(f"  p{p:<3} {int(np.percentile(sizes, p))}")

stats_chunker = make_chunker(CFG)
t0 = time.time()
n_chunks = np.array([len(stats_chunker.chunk(t)) for t in doc_text.values()])
print(f"\nChunked {len(n_chunks)} docs in {time.time() - t0:.1f}s with budget {CFG.max_chunk_chars}")
print(f"  chunks per doc: mean {n_chunks.mean():.2f} | median {np.median(n_chunks):.0f} | max {n_chunks.max()}")
print(f"  docs that stay 1 chunk: {100 * (n_chunks == 1).mean():.1f}%")
print(f"  docs with a tree-sitter ERROR node: {stats_chunker.stats['docs_with_parse_errors']} "
      f"({100 * stats_chunker.stats['docs_with_parse_errors'] / len(n_chunks):.2f}%)")
incomplete = [d for d, t in doc_text.items() if not code_is_complete(t, stats_chunker.chunk(t))]
print(f"  docs where the chunks miss some code: {len(incomplete)}" + (f"  e.g. {incomplete[:5]}" if incomplete else "  (good)"))

buckets = Counter("1" if k == 1 else "2" if k == 2 else "3-4" if k <= 4 else "5-9" if k <= 9 else "10+" for k in n_chunks)
print("  distribution:", {b: buckets.get(b, 0) for b in ["1", "2", "3-4", "5-9", "10+"]})

qid0 = next(iter(qrels))
print("\nExample query:\n", query_text[qid0][:400], "...")
print("\nIts correct solution:\n", doc_text[next(iter(qrels[qid0]))][:400])

## Step 7b: How long are the queries (and chunks) in tokens?
Counts tokens with CodeRankEmbed's own tokenizer (query prefix included), so you can pick `max_seq_length` before the long run. Anything above `max_seq_length` is cut off from the **end**. CodeRankEmbed cannot go above 8192 tokens.

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(CFG.model_name, trust_remote_code=True)
tok.model_max_length = 10**9                      # only counting here, so silence the length warning

q_ids = list(query_text)
q_lens = np.array([len(x) for x in tok([CFG.query_prefix + query_text[q] for q in q_ids])["input_ids"]])

print(f"Queries: {len(q_lens)}")
print(f"  mean {q_lens.mean():.0f} | median {np.median(q_lens):.0f} | "
      + " | ".join(f"p{p} {np.percentile(q_lens, p):.0f}" for p in (75, 90, 95, 99))
      + f" | max {q_lens.max()}")
longest = q_ids[int(q_lens.argmax())]
print(f"  longest query: id={longest}, {q_lens.max()} tokens, {len(query_text[longest].split())} words")

def cutoff_table(lens, name, limits=(512, 1024, 2048, 4096, 8192)):
    print(f"\n{name}: how many get cut at each max_seq_length")
    print(f"  {'limit':>6} {'cut':>7} {'% cut':>7} {'avg tokens lost (cut ones)':>28}")
    for L in limits:
        cut = lens > L
        lost = (lens[cut] - L).mean() if cut.any() else 0
        print(f"  {L:>6} {cut.sum():>7} {100 * cut.mean():>6.1f}% {lost:>28.0f}")

cutoff_table(q_lens, "Queries")

# What does the cut-off part look like? Show the end of a median-length query.
median_q = q_ids[int(np.argsort(q_lens)[len(q_lens) // 2])]
print(f"\nEnd of a median-length query ({median_q}), i.e. the part that is cut first:\n")
print("...", query_text[median_q][-600:])

# Chunks: check that no chunk is longer than max_seq_length (e.g. a huge single node).
chunk_texts = [c["text"] for t in doc_text.values() for c in make_chunker(CFG).chunk(t)]
c_lens = np.array([len(x) for x in tok(chunk_texts)["input_ids"]])
print(f"\nChunks: {len(c_lens)} | mean {c_lens.mean():.0f} | p99 {np.percentile(c_lens, 99):.0f} | max {c_lens.max()} tokens")
cutoff_table(c_lens, "Chunks")

## Step 8: The MTEB encoder (`PrePostPipelineEncoder`)

MTEB calls `encode()` twice: once for queries (`PromptType.query`) and once for documents (`PromptType.document`), and it must get back **exactly one vector per input**.

- **Queries → pre:** add the CodeRankEmbed prefix → embed.
- **Documents → pre:** AST-chunk every document → embed all chunks. **Post:** length-weighted mean of each document's chunk vectors → normalize → one vector per document.

In [ ]:
class PrePostPipelineEncoder(AbsEncoder):
    def __init__(self, cfg: Config = CFG, use_chunking: bool = True, model: SentenceTransformer | None = None):
        self.cfg = cfg
        self.use_chunking = use_chunking

        if model is None:                                   # load once, can be shared between runs
            model = SentenceTransformer(cfg.model_name, trust_remote_code=True, device=DEVICE)
            model.max_seq_length = cfg.query_max_seq_length
            if cfg.fp16 and DEVICE == "cuda":
                model.half()
        self.model = model
        self.chunker = make_chunker(cfg)
        self.doc_chunk_counts: list[int] = []
        self.docs_encoded = 0                               # counts documents actually encoded (not from cache)

        tag = (f"cast-c{cfg.max_chunk_chars}-anc{int(cfg.add_ancestor_header)}-imp{int(cfg.add_imports_header)}"
               if use_chunking else "no-chunking")
        self.mteb_model_meta = ModelMeta.create_empty(overwrites=dict(
            name=f"local/{cfg.model_name.split('/')[-1]}-{tag}",
            revision=tag,
            languages=["eng-Latn", "python-Code"],
            framework=["Sentence Transformers", "PyTorch"],
            similarity_fn_name=ScoringFunction.COSINE,
            embed_dim=(getattr(model, "get_embedding_dimension", None)
                       or model.get_sentence_embedding_dimension)(),
            max_tokens=cfg.query_max_seq_length,
            open_weights=True,
            use_instructions=False,
            reference=f"https://huggingface.co/{cfg.model_name}",
        ))

    def _embed(self, texts: list[str], batch_size: int, max_len: int) -> np.ndarray:
        self.model.max_seq_length = max_len               # queries and documents use different limits
        emb = self.model.encode(texts, batch_size=batch_size, normalize_embeddings=True,
                                convert_to_numpy=True, show_progress_bar=True)
        if DEVICE == "cuda":
            torch.cuda.empty_cache()                      # give the memory back before the next encode call
        return emb.astype(np.float32)

    def encode(self, inputs, *, task_metadata, hf_split, hf_subset, prompt_type=None, **kwargs) -> np.ndarray:
        batch_size = kwargs.get("batch_size", self.cfg.batch_size)
        texts = [t for batch in inputs for t in batch["text"]]

        # ---- queries: prefix only ----
        if prompt_type == PromptType.query:
            return self._embed([self.cfg.query_prefix + t for t in texts], self.cfg.query_batch_size,
                               self.cfg.query_max_seq_length)

        # ---- documents without chunking (baseline) ----
        self.docs_encoded += len(texts)
        if not self.use_chunking:
            return self._embed(texts, batch_size, self.cfg.doc_max_seq_length)

        # ---- documents: PRE = AST chunking ----
        chunk_texts, owner, weights = [], [], []
        for doc_idx, text in enumerate(texts):
            chunks = self.chunker.chunk(text)
            self.doc_chunk_counts.append(len(chunks))
            for c in chunks:
                chunk_texts.append(c["text"])
                owner.append(doc_idx)
                weights.append(max(c["size"], 1))
        chunk_emb = self._embed(chunk_texts, batch_size, self.cfg.doc_max_seq_length)

        # ---- POST = length-weighted mean of chunk vectors -> one vector per document ----
        owner = np.asarray(owner)
        weights = np.asarray(weights, dtype=np.float32)
        doc_emb = np.zeros((len(texts), chunk_emb.shape[1]), dtype=np.float32)
        np.add.at(doc_emb, owner, chunk_emb * weights[:, None])
        doc_emb /= np.clip(np.linalg.norm(doc_emb, axis=1, keepdims=True), 1e-12, None)
        return doc_emb

## Step 9: Run the evaluation and write `appsretrieval_results.json`
Same structure as the provided template, plus checkpoints saved to Drive:
1. **Embedding cache**: query and document vectors are saved as soon as each is encoded. Re-running this cell after a crash reloads them (seconds instead of minutes).
2. **Rankings** (`prediction_folder`) for Step 10.
3. **MTEB's own result copy** in `mteb_results/`.
4. **The result object** is stored in `RESULTS` before the JSON file is written, and the JSON is written safely (temp file, then rename).

The first run takes several minutes on a T4.

In [ ]:
RESULTS = {}      # tag -> MTEB TaskResult, kept in memory even if saving the file fails


def save_result_json(task_result, path) -> None:
    result_dict = task_result.to_dict()
    # to_dict() keeps "date" as a Python datetime, which json can't write.
    # Convert it the same way MTEB's own TaskResult.to_disk() does: to a Unix timestamp.
    if isinstance(result_dict.get("date"), datetime):
        result_dict["date"] = result_dict["date"].timestamp()
    tmp = Path(str(path) + ".tmp")                     # write to a temp file first, then rename:
    with open(tmp, "w") as f:                          # a crash mid-write never leaves a half-written file
        json.dump(result_dict, f, indent=2, default=str)
    tmp.replace(path)


def run_eval(model: PrePostPipelineEncoder, results_file: str, prediction_folder: str) -> dict:
    tag = model.mteb_model_meta.revision
    # Checkpoint 1: embeddings. Each run setting (tag) gets its own cache folder, keyed by the input text.
    # If a later step crashes, re-running this cell reloads the vectors instead of encoding again.
    cache_dir = Path(CFG.emb_cache_dir) / CFG.cache_version / tag
    cached_model = CachedEmbeddingWrapper(model, cache_path=cache_dir)

    task = mteb.get_task(CFG.task_name)
    t0 = time.time()
    result = mteb.evaluate(
        cached_model,
        [task],
        encode_kwargs={"batch_size": CFG.batch_size},
        overwrite_strategy="always",                   # always re-score; encoding is skipped if cached
        prediction_folder=prediction_folder,           # Checkpoint 2: rankings (on Drive)
        cache=mteb.ResultCache(cache_path=RUN_DIR / "mteb_results"),   # Checkpoint 3: MTEB's own copy
    )
    task_result = list(result.task_results)[0]
    RESULTS[tag] = task_result                         # Checkpoint 4: kept in memory before any file writing
    save_result_json(task_result, results_file)

    scores = task_result.scores["test"][0]
    print(f"\n[{tag}] done in {(time.time() - t0) / 60:.1f} min -> {results_file}")
    for k in ("ndcg_at_10", "mrr_at_10", "recall_at_1", "recall_at_10", "recall_at_100"):
        if k in scores:
            print(f"  {k:<14} {scores[k]:.4f}")
    if model.docs_encoded == 0:
        print("  documents loaded from the embedding cache (nothing re-encoded)")
    elif model.doc_chunk_counts:
        c = np.array(model.doc_chunk_counts)
        print(f"  docs encoded: {len(c)} | chunks: {c.sum()} | chunks/doc: {c.mean():.2f}")
    else:
        print(f"  docs encoded: {model.docs_encoded} (whole documents, no chunking)")
    return scores


def main() -> tuple[PrePostPipelineEncoder, dict]:
    model = PrePostPipelineEncoder()
    scores = run_eval(model, CFG.results_file, CFG.prediction_folder)
    return model, scores


encoder, main_scores = main()

## Step 10: Qualitative analysis
Looks at the saved rankings: where the correct solution landed, whether documents with many chunks do worse, and a few successes and failures with their chunks.

In [ ]:
def load_preds(prediction_folder: str) -> dict:
    with open(Path(prediction_folder) / f"{CFG.task_name}_predictions.json") as f:
        pred_file = json.load(f)
    pred_subset = next(k for k in pred_file if k != "mteb_model_meta")
    return pred_file[pred_subset]["test"]             # {query_id: {doc_id: score}}


def gold_ranks(preds: dict) -> dict:
    # Rank of the correct doc for every query (None = not in the saved top-k list).
    out = {}
    for qid, rel in qrels.items():
        gold = {d for d, s in rel.items() if s > 0}
        ranked = sorted(preds.get(qid, {}).items(), key=lambda x: -x[1])
        out[qid] = next((i + 1 for i, (d, _) in enumerate(ranked) if d in gold), None)
    return out


# Bucket = number of chunks the correct doc gets with the MAIN config (same grouping for every run).
qa_chunker = make_chunker(CFG)
BUCKETS = ["1", "2", "3-4", "5+"]
def _bucket(k): return "1" if k == 1 else "2" if k == 2 else "3-4" if k <= 4 else "5+"
GOLD_BUCKET = {qid: _bucket(len(qa_chunker.chunk(doc_text[next(iter(rel))]))) for qid, rel in qrels.items()}


def mrr_by_bucket(ranks: dict) -> dict:
    per = {}
    for qid, r in ranks.items():
        per.setdefault(GOLD_BUCKET[qid], []).append(0.0 if r is None or r > 10 else 1.0 / r)
    return {b: (float(np.mean(v)), len(v)) for b, v in per.items()}


preds = load_preds(CFG.prediction_folder)
gold_rank = gold_ranks(preds)

def ranking(qid):
    return sorted(preds.get(qid, {}).items(), key=lambda x: -x[1])

ranks = list(gold_rank.values())
n = len(ranks)
print(f"Queries: {n}")
print(f"  correct doc at rank 1:   {sum(r == 1 for r in ranks) / n:.1%}")
print(f"  correct doc in top 10:   {sum(r is not None and r <= 10 for r in ranks) / n:.1%}")
print(f"  not in saved top-k list: {sum(r is None for r in ranks) / n:.1%}")

# Does the number of chunks of the correct doc matter? (Compare with the no-chunking run in Step 11.)
print("\nMRR@10 by number of chunks in the correct document:")
main_buckets = mrr_by_bucket(gold_rank)
for b in BUCKETS:
    if b in main_buckets:
        m, cnt = main_buckets[b]
        print(f"  {b:>4} chunks: MRR@10 = {m:.3f}  ({cnt} queries)")

def show_example(qid, top=3):
    gold = next(iter(qrels[qid]))
    print("=" * 100)
    print(f"QUERY {qid} | correct doc {gold} at rank {gold_rank[qid]}")
    print(query_text[qid][:350].replace("\n", " "), "...")
    for i, (d, s) in enumerate(ranking(qid)[:top]):
        mark = "  <-- correct" if d == gold else ""
        print(f"  #{i + 1} {d} score={s:.4f}{mark}")
    top1 = ranking(qid)[0][0]
    print(f"\n  Chunks of the #1 document ({top1}):")
    print_chunks(qa_chunker.chunk(doc_text[top1]), max_lines=6)

successes = [q for q, r in gold_rank.items() if r == 1]
failures = [q for q, r in gold_rank.items() if r is None or r > 10]
for q in random.sample(successes, min(2, len(successes))):
    show_example(q)
for q in random.sample(failures, min(2, len(failures))):
    show_example(q)

## Step 11 (optional): Ablations
Set `RUN_ABLATIONS = True` to compare against (a) the same chunking **without** ancestor headers and (b) **no chunking** (whole document embedded). The model is shared, so it isn't loaded again. Each run takes about as long as Step 9. The submission file from Step 9 is not touched.

In [ ]:
RUN_ABLATIONS = False

if RUN_ABLATIONS:
    rows = {"AST chunks + ancestor header (main)": main_scores}

    no_header = PrePostPipelineEncoder(replace(CFG, add_ancestor_header=False), model=encoder.model)
    rows["AST chunks, no header"] = run_eval(no_header, str(RUN_DIR / "ablation_no_header.json"), str(RUN_DIR / "predictions" / "no_header"))

    no_chunk = PrePostPipelineEncoder(CFG, use_chunking=False, model=encoder.model)
    rows["No chunking (whole document)"] = run_eval(no_chunk, str(RUN_DIR / "ablation_no_chunking.json"), str(RUN_DIR / "predictions" / "no_chunking"))

    print(f"\n{'setting':<38}{'nDCG@10':>10}{'MRR@10':>10}{'R@10':>10}")
    for name, s in rows.items():
        print(f"{name:<38}" + "".join(f"{s.get(k, float('nan')):>10.4f}" for k in ("ndcg_at_10", "mrr_at_10", "recall_at_10")))

## Step 11b: Fair comparison by number of chunks
Step 10 showed lower scores for queries whose correct document has more chunks. But those are also *longer solutions*, which may just be harder problems. This compares **every run on the same query groups** (grouped by the chunk count under the main config). It only reads the saved rankings, nothing is re-evaluated.

Documents in group "1" get the same vector with or without chunking, so a real effect of chunking can only show up in groups 2+.

In [ ]:
folders = {"AST + header (main)": CFG.prediction_folder,
           "AST, no header": str(RUN_DIR / "predictions" / "no_header"),
           "No chunking": str(RUN_DIR / "predictions" / "no_chunking")}
tables = {name: mrr_by_bucket(gold_ranks(load_preds(f))) for name, f in folders.items()
          if (Path(f) / f"{CFG.task_name}_predictions.json").exists()}

print(f"MRR@10 by chunks in the correct doc\n{'setting':<24}" + "".join(f"{b + ' ch':>10}" for b in BUCKETS))
for name, t in tables.items():
    print(f"{name:<24}" + "".join(f"{t.get(b, (float('nan'), 0))[0]:>10.4f}" for b in BUCKETS))
print(f"{'queries':<24}" + "".join(f"{main_buckets.get(b, (0, 0))[1]:>10}" for b in BUCKETS))

# How often does the ancestor header actually appear?
all_chunks = [c for t in doc_text.values() for c in make_chunker(CFG).chunk(t)]
with_header = sum(1 for c in all_chunks if c["ancestors"])
print(f"\nChunks with an ancestor header: {with_header} of {len(all_chunks)} ({100 * with_header / len(all_chunks):.1f}%)")

## Step 13: Failure analysis: why is the correct solution ranked low?
Five checks, from automatic to manual:

1. **Groups:** split queries into *good* (rank 1-10), *mid* (11-100) and *bad* (>100).
2. **Features:** measure simple properties of each query and its correct document (length, math, boilerplate, digits, Python 2, word overlap, chunk count) and see which ones differ between groups.
3. **Scores:** is the correct document scored *low* (the model doesn't see the match), or scored *high but crowded out* by many others?
4. **Hubs:** documents that show up in the top 10 for a huge number of queries (generic code that "matches everything").
5. **Label noise:** queries with an identical twin (same problem text) but a different gold solution. Then a "wrong" top result may actually be a correct solution.

Finally, a sample of bad queries is saved as a CSV for **manual labelling**, which is where you'll find the real causes.

Set `ANALYSIS_FOLDER` to the run you want to analyse. The no-chunking run is the cleanest for general causes, because chunking effects aren't mixed in.

In [ ]:
import re
import pandas as pd

ANALYSIS_FOLDER = str(RUN_DIR / "predictions" / "no_chunking")   # or CFG.prediction_folder
if not (Path(ANALYSIS_FOLDER) / f"{CFG.task_name}_predictions.json").exists():
    ANALYSIS_FOLDER = CFG.prediction_folder
print("Analysing:", ANALYSIS_FOLDER)

a_preds = load_preds(ANALYSIS_FOLDER)
a_rank = gold_ranks(a_preds)
N_SAVED = max(len(v) for v in a_preds.values())               # length of the saved ranking per query

# ---------- features ----------
fa_chunker = make_chunker(CFG)
IO_LINE = re.compile(r"input\(|stdin|print\b|map\(int|\.split\(")
PY2 = re.compile(r"raw_input|xrange|^\s*print\s+[^\s(=]", re.M)
WORD = re.compile(r"[A-Za-z_]{3,}")
STOP = {"the", "and", "for", "you", "are", "that", "with", "this", "each", "from", "print", "input",
        "int", "map", "list", "range", "len", "return", "def", "else", "elif", "while", "import", "not", "sys"}

def query_features(q):
    body = q.split("-----Example")[0]                         # APPS puts examples at the end
    return {
        "q_words": len(q.split()),
        "q_math_marks": q.count("$"),                          # LaTeX math
        "q_has_image": int("[Image]" in q),
        "q_story_words": len(body.split()),                    # text before the examples
    }

def doc_features(code):
    lines = [l for l in code.split("\n") if l.strip() and not l.strip().startswith("#")]
    nonws = max(nonws_len(code), 1)
    return {
        "d_nonws_chars": nonws,
        "d_lines": len(lines),
        "d_chunks": len(fa_chunker.chunk(code)),
        "d_io_line_share": sum(bool(IO_LINE.search(l)) for l in lines) / max(len(lines), 1),
        "d_digit_share": sum(ch.isdigit() for ch in code) / nonws,     # lookup tables
        "d_python2": int(bool(PY2.search(code))),
        "d_has_def": int("def " in code),
    }

def overlap(q, code):
    # share of the doc's meaningful identifiers that also appear as words in the query
    qw = {w.lower() for w in WORD.findall(q)}
    dw = {w.lower() for w in WORD.findall(code)} - STOP
    return len(dw & qw) / max(len(dw), 1)

rows = []
for qid, rel in qrels.items():
    gold = next(iter(rel))
    ranked = sorted(a_preds.get(qid, {}).items(), key=lambda x: -x[1])
    r = a_rank[qid]
    rows.append({
        "qid": qid, "gold": gold,
        "rank": r if r is not None else N_SAVED + 1,           # not in saved list -> worst rank
        "gold_score": a_preds.get(qid, {}).get(gold, np.nan),
        "top1_score": ranked[0][1] if ranked else np.nan,
        "top1_doc": ranked[0][0] if ranked else None,
        "word_overlap": overlap(query_text[qid], doc_text[gold]),
        **query_features(query_text[qid]),
        **doc_features(doc_text[gold]),
    })
fa = pd.DataFrame(rows)
fa["group"] = pd.cut(fa["rank"], [0, 10, 100, np.inf], labels=["good (1-10)", "mid (11-100)", "bad (>100)"])
print(fa["group"].value_counts().sort_index().to_string(), "\n")

# ---------- check 2: which features differ between groups? ----------
feat_cols = ["q_words", "q_story_words", "q_math_marks", "q_has_image", "word_overlap",
             "d_nonws_chars", "d_lines", "d_chunks", "d_io_line_share", "d_digit_share", "d_python2", "d_has_def"]
print("Median (mean for 0/1 features) per group:")
summary = pd.concat([
    fa.groupby("group", observed=True)[[c for c in feat_cols if not c.startswith(("q_has", "d_python2", "d_has"))]].median(),
    fa.groupby("group", observed=True)[["q_has_image", "d_python2", "d_has_def"]].mean(),
], axis=1).T
print(summary.round(3).to_string(), "\n")

print("Spearman correlation with rank (positive = feature goes with WORSE rank):")
corr = fa[feat_cols + ["rank"]].corr(method="spearman")["rank"].drop("rank")
print(corr.reindex(corr.abs().sort_values(ascending=False).index).round(3).to_string())

In [ ]:
# ---------- check 3: low score, or crowded out? ----------
print("Scores (median):")
print(fa.groupby("group", observed=True)[["gold_score", "top1_score"]].median().round(4).to_string())
bad = fa[fa["group"] == "bad (>100)"]
print(f"\nBad queries: gold score is {np.nanmedian(bad['top1_score'] - bad['gold_score']):.4f} below the top-1 score (median).")
print("Small gap + bad rank  -> many docs score almost the same (crowding: the model can't tell solutions apart).")
print("Large gap             -> the model doesn't see the match at all (query-code mismatch).\n")

# ---------- check 4: hub documents ----------
top10_counts = Counter(d for qid in qrels for d, _ in sorted(a_preds.get(qid, {}).items(), key=lambda x: -x[1])[:10])
hubs = top10_counts.most_common(10)
print("Documents appearing in the most top-10 lists (expected for a random doc: ~"
      f"{10 * len(qrels) / len(doc_text):.1f}):")
for d, c in hubs:
    f = doc_features(doc_text[d])
    print(f"  {d:<8} in {c:>4} top-10s | {f['d_lines']:>4} lines | io-share {f['d_io_line_share']:.2f} | "
          f"first line: {doc_text[d].strip().splitlines()[0][:60]!r}")
hub_set = {d for d, c in top10_counts.items() if c >= np.percentile(list(top10_counts.values()), 99)}
print(f"\nBad queries whose #1 result is a top-1% hub: {bad['top1_doc'].isin(hub_set).mean():.1%} "
      f"(good queries: {fa[fa['group'] == 'good (1-10)']['top1_doc'].isin(hub_set).mean():.1%})\n")

# ---------- check 5: label noise from duplicate problems ----------
def norm(t): return " ".join(t.split()).lower()
twins = {}
for qid in qrels:
    twins.setdefault(norm(query_text[qid]), []).append(qid)
twin_groups = [g for g in twins.values() if len(g) > 1]
print(f"Queries with an identical twin query: {sum(len(g) for g in twin_groups)} in {len(twin_groups)} groups")

rescued = 0
for g in twin_groups:
    golds = {next(iter(qrels[q])) for q in g}
    for q in g:
        if a_rank[q] is None or a_rank[q] > 10:
            top10 = [d for d, _ in sorted(a_preds.get(q, {}).items(), key=lambda x: -x[1])[:10]]
            if golds & set(top10):
                rescued += 1           # a twin's gold solution IS in the top 10 -> probably a correct answer
print(f"Queries counted as failures although a twin's gold solution is in their top 10: {rescued}")

In [ ]:
# ---------- manual labelling sheet ----------
N_LABEL = 30
sample = bad.sample(min(N_LABEL, len(bad)), random_state=0)
sheet = pd.DataFrame({
    "qid": sample["qid"],
    "gold_rank": sample["rank"],
    "gold_score": sample["gold_score"].round(4),
    "top1_score": sample["top1_score"].round(4),
    "query_start": [query_text[q][:500] for q in sample["qid"]],
    "gold_code": [doc_text[d][:800] for d in sample["gold"]],
    "top1_code": [doc_text[d][:800] for d in sample["top1_doc"]],
    "cause": "",          # fill in by hand, e.g. one of the labels in the notes below
    "notes": "",
})
sheet_path = RUN_DIR / "failure_labelling_sheet.csv"
sheet.to_csv(sheet_path, index=False)
print(f"Saved {len(sheet)} bad queries for manual labelling -> {sheet_path}")

**Suggested `cause` labels for the sheet** (pick one per row, add your own if needed):
- `boilerplate`: the gold solution is mostly input/output code; little problem-specific logic.
- `story_query`: the problem is wrapped in a long story; the actual task is hard to extract.
- `math_heavy`: the key information is in formulas (`$...$`).
- `short_trick`: the solution is a tiny formula/trick with no link to the story words.
- `lookup_table`: the solution is mostly hard-coded data.
- `similar_problem`: the #1 result solves a very similar problem (not truly wrong in meaning).
- `duplicate_label`: the #1 result is actually a correct solution for this problem.
- `other`

Once 30 rows are labelled, count the causes. That count tells you which fix to try: e.g. many `story_query` → query rewriting/GAR; many `similar_problem`/`duplicate_label` → benchmark noise, not a model problem.

## Step 14: Agent first pass: where does the correct solution rank after the LLM interprets the query?

For a sample of queries, an LLM (via API) reads the problem **once** and writes:
- **task**: what must be computed, in 1-2 plain sentences, without the story;
- **algorithms**: 2-3 short technique phrases (e.g. "sort and take prefix sums");
- **sketch**: a short Python solution in typical competitive-programming style.

Each of these becomes a separate search over the corpus (CodeRankEmbed, whole documents, no chunking). We record **where the correct solution ranks** in each search, and whether it is anywhere in the **candidate pool** (the union of the top-K results of all agent searches). No loop, no review, no code execution yet: this measures only the first pass.

The sample is stratified by the baseline result (good / mid / bad), so we can see whether the **bad** group gets pulled into the pool.

**Setup:** add your API key in Colab's *Secrets* panel (key icon on the left) as `ANTHROPIC_API_KEY` and allow notebook access. Requires Steps 2, 2b, 3, 7 and 10 to have run in this session.

**Caveat for the write-up:** APPS problems are public, so the LLM may recognise a problem and recall its solution. That can inflate the `sketch` results in particular.

In [ ]:
!pip install -q anthropic

import concurrent.futures as cf
import anthropic
import pandas as pd

# ---------- agent settings ----------
AGENT_MODEL = "claude-haiku-4-5-20251001"     # cheaper and fast; "claude-sonnet-5" is stronger
N_PER_GROUP = 100                             # sample size per group (good / mid / bad) -> 300 queries
POOL_KS = (10, 20, 50, 100)                   # candidate-pool sizes to report (top-K per search)
MAX_WORKERS = 4                               # parallel API calls
AGENT_DIR = RUN_DIR / "agent_first_pass"
AGENT_DIR.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import userdata
    api_key = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    api_key = os.environ.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic(api_key=api_key, max_retries=5)   # retries rate limits / server errors itself

# ---------- stratified sample from the no-chunking baseline ----------
base_folder = RUN_DIR / "predictions" / "no_chunking"
if not (base_folder / f"{CFG.task_name}_predictions.json").exists():
    base_folder = Path(CFG.prediction_folder)
base_rank = gold_ranks(load_preds(str(base_folder)))

def base_group(r):
    return "good" if r is not None and r <= 10 else "mid" if r is not None and r <= 100 else "bad"

rng = random.Random(0)
by_group = {"good": [], "mid": [], "bad": []}
for qid, r in base_rank.items():
    by_group[base_group(r)].append(qid)
sample_qids = [q for g in ("good", "mid", "bad")
               for q in rng.sample(sorted(by_group[g]), min(N_PER_GROUP, len(by_group[g])))]
print(f"Baseline: {base_folder.name} | sample: {len(sample_qids)} queries "
      + str({g: min(N_PER_GROUP, len(v)) for g, v in by_group.items()}))

In [ ]:
# ---------- Interpret each query with the LLM (answers are cached on Drive) ----------
SYSTEM_PROMPT = (
    "You help a code search engine find the Python solution to a competitive programming problem.\n"
    "Read the problem and return ONLY a JSON object, no other text, with these keys:\n"
    '  "task": 1-2 plain sentences stating exactly what must be computed or output. '
    "No story, no character names.\n"
    '  "algorithms": a list of 2-3 short phrases, each naming a technique and its key operation '
    '(e.g. "sort the array and take prefix sums", "BFS on a grid from the start cell").\n'
    '  "sketch": a short Python 3 solution (at most 15 lines) in typical competitive-programming style: '
    "read stdin with input() and map(int, input().split()), short variable names, print the answer."
)

cache_path = AGENT_DIR / f"interpretations_{AGENT_MODEL}.jsonl"
interp = {}
if cache_path.exists():
    with open(cache_path) as f:
        for line in f:
            rec = json.loads(line)
            interp[rec["qid"]] = rec

def parse_json(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1].rsplit("```", 1)[0]
    start, end = text.find("{"), text.rfind("}")
    obj = json.loads(text[start:end + 1])
    algos = obj.get("algorithms", [])
    return {"task": str(obj["task"]),
            "algorithms": [str(a) for a in (algos if isinstance(algos, list) else [algos])][:3],
            "sketch": str(obj.get("sketch", ""))}

def interpret(qid):
    messages = [{"role": "user", "content": query_text[qid]}]
    text = ""
    for attempt in range(2):
        resp = client.messages.create(model=AGENT_MODEL, max_tokens=900, temperature=0,
                                      system=SYSTEM_PROMPT, messages=messages)
        text = "".join(b.text for b in resp.content if b.type == "text")
        try:
            out = parse_json(text)
            return {"qid": qid, **out,
                    "in_tokens": resp.usage.input_tokens, "out_tokens": resp.usage.output_tokens}
        except Exception:
            messages = messages + [{"role": "assistant", "content": text},
                                   {"role": "user", "content": "That was not valid JSON. Return only the JSON object."}]
    return {"qid": qid, "task": "", "algorithms": [], "sketch": "", "error": text[:300]}

todo = [q for q in sample_qids if q not in interp or not interp[q]["task"]]   # also retries earlier failures
print(f"Cached: {len(sample_qids) - len(todo)} | to call: {len(todo)}")
t0 = time.time()
with cf.ThreadPoolExecutor(MAX_WORKERS) as pool, open(cache_path, "a") as f:
    futures = {pool.submit(interpret, q): q for q in todo}
    for i, fut in enumerate(cf.as_completed(futures), 1):
        try:
            rec = fut.result()
        except Exception as e:                       # e.g. a wrong API key: stop with a clear message
            raise RuntimeError(f"API call failed for {futures[fut]}: {e}") from e
        interp[rec["qid"]] = rec
        f.write(json.dumps(rec) + "\n")              # checkpoint: every answer is saved immediately
        f.flush()
        if i % 25 == 0 or i == len(todo):
            print(f"  {i}/{len(todo)} done ({time.time() - t0:.0f}s)")

failed = [q for q in sample_qids if not interp[q]["task"]]
tok_in = sum(interp[q].get("in_tokens", 0) for q in sample_qids)
tok_out = sum(interp[q].get("out_tokens", 0) for q in sample_qids)
print(f"Failed to parse: {len(failed)} | tokens for this sample: {tok_in:,} in / {tok_out:,} out")

ex = interp[sample_qids[-1]]
print("\nExample interpretation:\n  task:", ex["task"], "\n  algorithms:", ex["algorithms"], "\n  sketch:\n" + ex["sketch"])

In [ ]:
# ---------- Embed the corpus once (whole documents), cached on Drive ----------
if "encoder" in globals():
    st_model = encoder.model
else:
    st_model = SentenceTransformer(CFG.model_name, trust_remote_code=True, device=DEVICE)
    if CFG.fp16 and DEVICE == "cuda":
        st_model.half()

def embed(texts, max_len, batch_size):
    st_model.max_seq_length = max_len
    return st_model.encode(texts, batch_size=batch_size, normalize_embeddings=True,
                           convert_to_numpy=True, show_progress_bar=len(texts) > 200).astype(np.float32)

doc_ids = sorted(doc_text)
emb_path = AGENT_DIR / "corpus_whole_doc_emb.npy"
ids_path = AGENT_DIR / "corpus_whole_doc_ids.json"
if emb_path.exists() and ids_path.exists() and json.load(open(ids_path)) == doc_ids:
    D = np.load(emb_path)
    print("Loaded corpus embeddings from Drive:", D.shape)
else:
    D = embed([doc_text[d] for d in doc_ids], CFG.doc_max_seq_length, CFG.batch_size)
    np.save(emb_path, D)
    json.dump(doc_ids, open(ids_path, "w"))
    print("Encoded corpus:", D.shape)
doc_index = {d: i for i, d in enumerate(doc_ids)}

# ---------- The searches for every sampled query ----------
# name -> (text, is_code). Natural-language searches get CodeRankEmbed's query prefix; code searches don't.
def searches_for(qid):
    it = interp[qid]
    s = {"original": (query_text[qid], False)}
    if it["task"]:
        s["task"] = (it["task"], False)
        s["task+algorithms"] = (it["task"] + " " + " ".join(it["algorithms"]), False)
    for i, a in enumerate(it["algorithms"]):
        s[f"algorithm_{i + 1}"] = (a, False)
    if it["sketch"].strip():
        s["sketch"] = (it["sketch"], True)
    return s

flat = [(q, name, text, is_code) for q in sample_qids for name, (text, is_code) in searches_for(q).items()]
nl_idx = [i for i, x in enumerate(flat) if not x[3]]
code_idx = [i for i, x in enumerate(flat) if x[3]]
Q = np.zeros((len(flat), D.shape[1]), dtype=np.float32)
if nl_idx:
    Q[nl_idx] = embed([CFG.query_prefix + flat[i][2] for i in nl_idx], CFG.query_max_seq_length, CFG.query_batch_size)
if code_idx:
    Q[code_idx] = embed([flat[i][2] for i in code_idx], CFG.doc_max_seq_length, CFG.query_batch_size)

# ---------- Rank of the correct solution in every search + merged ranking + candidate pool ----------
RRF_K = 60
per_query = {}
for (q, name, _, _), sc in zip(flat, Q @ D.T):
    per_query.setdefault(q, {})[name] = sc

rows, pool_rows = [], []
for q, by_name in per_query.items():
    g = doc_index[next(iter(qrels[q]))]
    grp = base_group(base_rank[q])
    orders = {}
    for name, sc in by_name.items():
        rows.append({"qid": q, "group": grp, "search": name, "rank": int((sc > sc[g]).sum()) + 1})
        orders[name] = np.argsort(-sc)
    # merged ranking of all AGENT searches (reciprocal rank fusion; "original" left out)
    rrf = np.zeros(len(doc_ids))
    for name, order in orders.items():
        if name != "original":
            rrf[order] += 1.0 / (RRF_K + np.arange(1, len(order) + 1))
    if rrf.any():
        rows.append({"qid": q, "group": grp, "search": "merged (RRF)", "rank": int((rrf > rrf[g]).sum()) + 1})
    # candidate pool = union of the top-K of every agent search
    agent_orders = [o for n, o in orders.items() if n != "original"]
    rec = {"qid": q, "group": grp}
    for K in POOL_KS:
        pool = set(np.concatenate([o[:K] for o in agent_orders]).tolist()) if agent_orders else set()
        rec[f"in_pool@{K}"] = int(g in pool)
        rec[f"pool_size@{K}"] = len(pool)
        rec[f"orig_top{K}"] = int(g in set(orders["original"][:K].tolist()))
    pool_rows.append(rec)

res = pd.DataFrame(rows)
pool_df = pd.DataFrame(pool_rows)
res.to_csv(AGENT_DIR / f"first_pass_ranks_{AGENT_MODEL}.csv", index=False)

# Sanity check: our "original" search should reproduce the MTEB baseline ranks
orig = res[res["search"] == "original"].set_index("qid")["rank"]
match = np.mean([(orig[q] > 1000) if base_rank[q] is None else abs(orig[q] - base_rank[q]) <= 2 for q in orig.index])
print(f"Sanity check: 'original' rank within +-2 of the MTEB baseline for {match:.0%} of sampled queries")

In [ ]:
def summarise(r):
    return pd.Series({"median rank": r.median(), "R@1": (r <= 1).mean(), "R@10": (r <= 10).mean(),
                      "R@100": (r <= 100).mean(), "MRR@10": np.where(r <= 10, 1.0 / r, 0.0).mean()})

order = ["original", "task", "task+algorithms", "algorithm_1", "algorithm_2", "algorithm_3", "sketch", "merged (RRF)"]
present = [o for o in order if o in set(res["search"])]
GROUPS = ["good", "mid", "bad"]

print("1) Rank of the correct solution per search (all sampled queries)")
print(res.groupby("search")["rank"].apply(summarise).unstack().reindex(present).round(3).to_string(), "\n")

print("2) R@100 per search, by baseline group  (does the BAD group get pulled in?)")
t2 = res.pivot_table(index="search", columns="group", values="rank", aggfunc=lambda r: (r <= 100).mean())
print(t2.reindex(index=present, columns=GROUPS).round(3).to_string(), "\n")

print("3) Best rank over the agent searches (upper bound: if the agent always trusted its best search)")
best = res[~res["search"].isin(["original", "merged (RRF)"])].groupby("qid")["rank"].min()
bdf = pd.DataFrame({"rank": best, "group": [base_group(base_rank[q]) for q in best.index]})
print(bdf.groupby("group")["rank"].apply(lambda r: pd.Series({"R@10": (r <= 10).mean(), "R@100": (r <= 100).mean()}))
      .unstack().reindex(GROUPS).round(3).to_string(), "\n")

print("4) Candidate pool: is the correct solution anywhere in the union of the agent searches' top-K?")
rows4 = []
for g in GROUPS + ["all"]:
    sub = pool_df if g == "all" else pool_df[pool_df["group"] == g]
    for K in POOL_KS:
        rows4.append({"group": g, "K per search": K, "in agent pool": sub[f"in_pool@{K}"].mean(),
                      "in original top-K": sub[f"orig_top{K}"].mean(),
                      "avg pool size": sub[f"pool_size@{K}"].mean()})
print(pd.DataFrame(rows4).set_index(["group", "K per search"]).round(3).to_string())

In [ ]:
# ---------- A few examples from the BAD group ----------
wide = res.pivot_table(index="qid", columns="search", values="rank").reindex(columns=present)
bad_q = [q for q in sample_qids if base_group(base_rank[q]) == "bad"]
for q in random.Random(1).sample(bad_q, min(4, len(bad_q))):
    it = interp[q]
    print("=" * 100)
    print(f"{q} | gold rank per search: " + ", ".join(f"{k}={int(v)}" for k, v in wide.loc[q].dropna().items()))
    print("QUERY :", " ".join(query_text[q].split())[:300], "...")
    print("TASK  :", it["task"])
    print("ALGOS :", it["algorithms"])
    print("SKETCH:\n" + "\n".join(it["sketch"].splitlines()[:10]))
    print("GOLD  :\n" + "\n".join(doc_text[next(iter(qrels[q]))].strip().splitlines()[:10]))

## Step 12: Download the results file

In [ ]:
try:
    from google.colab import files
    files.download(CFG.results_file)
except ImportError:
    print(f"Not in Colab. The file is at: {Path(CFG.results_file).resolve()}")